# T3 tourniquet adapter

The progressive record for the fine-tune that `docs/training/t3_zeroshot.ipynb` decided:
zero-shot `cosmos-reason2-8b` fails the tourniquet coach at every grain, so a LoRA adapter
trains on the box. Shape per the plan: vision tower frozen, the multimodal projector
(merger) trained in full, rank-16 alpha-32 LoRA on every decoder linear. Scripts:
`t3_prep.py` (data), `t3_train.py` (training), `t3_serve.py` (serving), all in this
directory with runtime copies on the box under `~/flux-model/{bench,train}`.


## Data (built 2026-08-16)

`~/flux-model/train/data`: 2,724 train and 247 val examples over per-video-fps-correct
frames, split by video id with the zero-shot bench's eval videos excluded entirely.

| task | source | examples |
| --- | --- | --- |
| action grain, 5 procedures, both skill types | `regular/` + `JIT/` CSVs | 2,534 + 237 val |
| step grain, P05, six authored cues | same segments, remapped | 190 + 10 val |

The grounding and VQA tasks from the plan wait on the object class-id names and a
sampling pass over the 8.28M QA pairs; this run measures what step data alone buys.


## Environment (the GB10 fights back)

- The HF repo is gated and the box's token file is root-locked; the NIM container carries
  the full HF-format snapshot (`Qwen3VLForConditionalGeneration`), and
  `docker exec ... tar -chf -` streams it out in seconds where `docker cp` refuses the
  cache's out-of-tree symlinks.
- torch cu128 wheels crash at the first fused reduction: the jiterator asks nvrtc 12.8
  for the GB10's sm_121, which only CUDA 13 knows. `torch==2.13.0+cu130` aarch64 wheels
  fix it. torchvision must install too (the Qwen3-VL video processor imports it).
- The cosmos and nemotron NIMs hold most of the 121 GB unified memory, so the training
  chain stops them for the run and restarts them after; the coach and chat surfaces are
  down for the duration.
- One orphaned launch chain started a second training process against the same GPU;
  killed. A launch chain that can fire twice needs a lock, the same lesson the T3
  downloader learned.


## Run log

### Smoke — 30 steps, 2026-08-16: **fit confirmed**

44 optimizer steps/hour (batch 1, grad-accum 8), 32.9 GiB peak allocated, loss 0.82 at
step 10 falling to 0.31 by step 20. Trainable: LoRA over decoder linears plus the merger
in full.

### r1 — 2 epochs, 682 steps, launched 2026-08-16

lr 1e-4, 20-step warmup then cosine, clip 1.0, seed 20260816, checkpoints every 200
steps, `~15.5 h` at the smoke rate. Eval chain armed: on completion, `t3_serve.py` loads
`runs/r1/final` on port 30083 and the frozen bench reruns through the identical harness
(`t3_zeroshot.py --url ... --model cosmos-reason2-8b-t3`), all four set-grain
combinations. Numbers land here when it finishes.


## The bar (from the zero-shot notebook)

| set, grain | base model | pointer replay |
| --- | --- | --- |
| standard, step | 12/36 = 33.3% | 2/6 sessions reach S4+ |
| improvised, step | 54/123 = 43.9% | 6/14 sessions reach S4+ |
| standard, action | 1/36 = 2.8% | |
| improvised, action | 11/123 = 8.9% | |

The tile needs the pointer to reach the securing step in most sessions; per-clip step
accuracy in the 70s has delivered that for the knots.


## Serving and wiring

The deployment's cosmos NIM carries no adapter hooks, so `t3_serve.py` serves
base-plus-adapter as `cosmos-reason2-8b-t3` on port 30083. The server routes only the
tourniquet procedure there (`CoachKnot.model_env` / `url_env`, commits 793ffde and
39cd05a); `scripts/demo_up.sh` starts the process when the adapter exists and exports
the two env vars only on a healthy probe (b6cebb5). Every layer absent falls back to
the base model.


## Serving inference: what the tests measured (2026-08-16)

All numbers from the box's cosmos-reason2-8b NIM (port 30082, vLLM-backed,
`NIM_MAX_NUM_SEQS=4`, 25 percent GPU memory and KV cache) with the deployed coach
prompt, measured by the bench harness and a direct latency probe.

### Latency per classification request

| payload | cold | warm (prefix cached) |
| --- | --- | --- |
| 4 frames, 720p JPEG | 2.5 s | 1.0 s |
| 8 frames, 720p JPEG | 4.0 s | 1.1 s |

The prompt prefix (scaffold plus cue list, identical every clip of a session) stays in
the vLLM prefix cache, so a live coach session pays the cold cost once and roughly 1 to
2 s per clip after; the pointer rule needs 2 agreeing clips, so a step advance registers
in about 8 s of user time at the app's 3 s clip cadence.

### Payload rules (found the hard way)

- 8 base64 frames at native 1080p overflow the NIM request limit: HTTP 400. Downscale to
  720p; all 350+ bench requests then succeed.
- One frame per 2 s segment (the deployed 1 fps) starves temporal verbs; 4 evenly spaced
  frames minimum. `{"step": "S<n>"}` parsed from every response at temperature 0, zero
  unparseable across the corrected benches.

### Quality available today (zero-shot, no adapter)

Step grain, six authored cues: 33.3 percent standard, 43.9 percent improvised per clip
(chance 16.7); pointer within one step of truth two thirds of the time; 8 of 20 sessions
reach the securing step. Action grain (18 to 28 classes) sits at or below chance and is
not servable. Implication: the tourniquet coach serves today as narration, figures, and
coarse camera confirmation, and its camera verdicts must stay suggestions behind the
confirm chip, which is what the surface already does.

### Capacity

- Co-residency: NIMs plus VSS hold 86 to 96 of 121 GB; about 25 GB stays free. The
  adapter server (17 GB bf16 via transformers) fits that envelope, untested under load.
- Model load: about 85 s for the 8B snapshot from disk (transformers, bf16); the NIM
  reaches healthy in 2 to 4 minutes from `docker start`.
- Concurrency: the NIM caps at 4 in-flight sequences; the coach sends one request per
  clip per session, so 3 to 4 simultaneous coach sessions fit before queueing.
- Training co-residency does not exist: 32.9 GiB peak needs the NIMs stopped.
